# Haiku Example 1: CODEX + H&E Patch Visualization

This notebook visualizes preprocessed CODEX and H&E patches from the demo dataset:
- **3 example patches** from different tissue regions with multi-channel biomarker views
- **Whole-region mosaic** reconstructed from all patches of a single region

Data is loaded from `dataset/` — no external paths required.


In [ ]:
import pickle
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

HAIKU_ROOT = Path('/home/yancui/Haiku')
DATASET_ROOT = HAIKU_ROOT / 'dataset'
EXAMPLE_DIR = DATASET_ROOT / 'example_slices'
# Use ada-28577 for whole-region mosaic (largest region, 262 patches)
MOSAIC_REGION = 'ada-28577'


def norm01(arr):
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn + 1e-8)


print(f'Dataset : {DATASET_ROOT}')
print(f'Examples: {EXAMPLE_DIR}')
print(f'Mosaic  : {MOSAIC_REGION}')


In [ ]:
# Load the 3 example patches (one from each of 3 regions used in retrieval)
pkl_files = sorted(EXAMPLE_DIR.glob('*.pkl'))

examples = []
for pf in pkl_files:
    with open(pf, 'rb') as f:
        d = pickle.load(f)
    he = np.load(EXAMPLE_DIR / f'{pf.stem}.npy')
    region = pf.stem.split('_')[0] + '-' + pf.stem.split('_')[1].split('-')[0]
    # Extract region from filename: xxx-xxxxx_coords -> xxx-xxxxx
    region = '-'.join(pf.stem.split('_')[0:1]) if '-' in pf.stem.split('_')[0] else pf.stem.rsplit('_', 1)[0]
    examples.append({
        'name': pf.stem,
        'region': region,
        'codex': d['codex'],
        'markers': d['biomarker_name'],
        'he': he,
    })

print(f'Loaded {len(examples)} example patches:')
for e in examples:
    print(f'  {e["region"]} | {e["name"]} | {e["codex"].shape[0]} channels')


### Single-Patch Detail: H&E + Biomarker Channels

Each patch shows the registered H&E image alongside individual biomarker channels
and an RGB composite overlay.


In [ ]:
# Markers to display (common across panels)
row1_markers = ['DAPI', 'CD3e', 'CD8', 'PanCK', 'Ki67', 'CD20']
row2_markers = ['CD4', 'PDL1', 'HLA-DR', 'CollagenIV', 'EpCAM']

for p in examples:
    markers = p['markers']
    codex = p['codex']

    # Filter to markers available in this panel
    r1 = [m for m in row1_markers if m in markers]
    r2 = [m for m in row2_markers if m in markers]
    ncols = max(1 + len(r1), 1 + len(r2))
    fig, axes = plt.subplots(2, ncols, figsize=(3.2 * ncols, 6.5))

    # Row 1: H&E + individual channels
    axes[0, 0].imshow(p['he'])
    axes[0, 0].set_title('H&E', fontsize=10, fontweight='bold', color='#1565c0')
    for spine in axes[0, 0].spines.values():
        spine.set_edgecolor('#1565c0'); spine.set_linewidth(2)
    axes[0, 0].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    for j, m in enumerate(r1):
        axes[0, j + 1].imshow(codex[markers.index(m)], cmap='magma')
        axes[0, j + 1].set_title(m, fontsize=10, fontweight='bold')
        axes[0, j + 1].axis('off')
    for j in range(1 + len(r1), ncols):
        axes[0, j].axis('off')

    # Row 2: RGB composite + remaining channels
    ch_r = codex[markers.index('PanCK')].astype(np.float32) if 'PanCK' in markers else np.zeros_like(codex[0], dtype=np.float32)
    ch_g = codex[markers.index('CD3e')].astype(np.float32) if 'CD3e' in markers else np.zeros_like(codex[0], dtype=np.float32)
    ch_b = codex[markers.index('DAPI')].astype(np.float32) if 'DAPI' in markers else np.zeros_like(codex[0], dtype=np.float32)
    comp = np.stack([norm01(ch_r), norm01(ch_g), norm01(ch_b)], axis=-1)
    axes[1, 0].imshow(np.clip(comp, 0, 1))
    axes[1, 0].set_title('R=PanCK G=CD3e B=DAPI', fontsize=8, fontweight='bold', color='#2e7d32')
    for spine in axes[1, 0].spines.values():
        spine.set_edgecolor('#2e7d32'); spine.set_linewidth(2)
    axes[1, 0].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    for j, m in enumerate(r2):
        axes[1, j + 1].imshow(codex[markers.index(m)], cmap='magma')
        axes[1, j + 1].set_title(m, fontsize=10, fontweight='bold')
        axes[1, j + 1].axis('off')
    for j in range(1 + len(r2), ncols):
        axes[1, j].axis('off')

    fig.suptitle(f'{p["region"]}  |  {p["name"]}  |  {len(markers)} channels',
                 fontsize=11, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()


### Whole-Region Mosaic

Reconstruct the full tissue view from all patches of a single region.
The mosaic shows H&E, individual biomarker channels, and an RGB composite.


In [ ]:
codex_dir = DATASET_ROOT / 'codex_patches' / MOSAIC_REGION
he_dir = DATASET_ROOT / 'he_patches' / MOSAIC_REGION

codex_files = sorted(codex_dir.glob('*.pkl'))
print(f'CODEX patches: {len(codex_files)}')


def parse_coords(fname):
    m = re.search(r'_(\d+)-(\d+)-(\d+)-(\d+)', fname)
    if m:
        return int(m.group(1)), int(m.group(3))
    return 0, 0


patch_data = []
for pf in codex_files:
    r, c = parse_coords(pf.stem)
    with open(pf, 'rb') as f:
        d = pickle.load(f)
    hef = he_dir / f'{pf.stem}.npy'
    he_arr = np.load(hef) if hef.exists() else None
    patch_data.append({'r': r, 'c': c, 'codex': d['codex'],
                       'markers': d['biomarker_name'], 'he': he_arr})

rows = sorted(set(p['r'] for p in patch_data))
cols = sorted(set(p['c'] for p in patch_data))
row_idx = {r: i for i, r in enumerate(rows)}
col_idx = {c: i for i, c in enumerate(cols)}
PS = 256
nr, nc = len(rows), len(cols)

key_markers = ['DAPI', 'CD3e', 'CD8', 'PanCK', 'Ki67', 'CD20']
# Filter to markers available in this region
avail_markers = [m for m in key_markers if m in patch_data[0]['markers']]

he_mosaic = np.zeros((nr * PS, nc * PS, 3), dtype=np.uint8)
marker_mosaics = {m: np.zeros((nr * PS, nc * PS), dtype=np.float32) for m in avail_markers}

for p in patch_data:
    ri, ci = row_idx[p['r']], col_idx[p['c']]
    r0, r1 = ri * PS, (ri + 1) * PS
    c0, c1 = ci * PS, (ci + 1) * PS
    if p['he'] is not None:
        he_mosaic[r0:r1, c0:c1] = p['he']
    for m in avail_markers:
        if m in p['markers']:
            mi = p['markers'].index(m)
            marker_mosaics[m][r0:r1, c0:c1] = p['codex'][mi]

print(f'Mosaic: {nr} x {nc} patches ({nr * PS} x {nc * PS} px) | {len(avail_markers)} markers')


In [ ]:
ncols = 4
nrows = 2
fig, axes = plt.subplots(nrows, ncols, figsize=(24, 12))

# Row 1: H&E + first 3 markers
axes[0, 0].imshow(he_mosaic)
axes[0, 0].set_title('H&E (whole region)', fontsize=12, fontweight='bold', color='#1565c0')
for spine in axes[0, 0].spines.values():
    spine.set_edgecolor('#1565c0'); spine.set_linewidth(2)
axes[0, 0].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

for j, m in enumerate(avail_markers[:3]):
    axes[0, j + 1].imshow(marker_mosaics[m], cmap='magma')
    axes[0, j + 1].set_title(f'{m}', fontsize=12, fontweight='bold')
    axes[0, j + 1].axis('off')

# Row 2: remaining markers + RGB composite
for j, m in enumerate(avail_markers[3:]):
    axes[1, j].imshow(marker_mosaics[m], cmap='magma')
    axes[1, j].set_title(f'{m}', fontsize=12, fontweight='bold')
    axes[1, j].axis('off')

# RGB composite in last position
comp_r = norm01(marker_mosaics.get('PanCK', np.zeros((nr * PS, nc * PS))))
comp_g = norm01(marker_mosaics.get('CD3e', np.zeros((nr * PS, nc * PS))))
comp_b = norm01(marker_mosaics.get('DAPI', np.zeros((nr * PS, nc * PS))))
rgb = np.stack([comp_r, comp_g, comp_b], axis=-1)
axes[1, 3].imshow(np.clip(rgb, 0, 1))
axes[1, 3].set_title('Composite (R=PanCK, G=CD3e, B=DAPI)', fontsize=11,
                      fontweight='bold', color='#2e7d32')
for spine in axes[1, 3].spines.values():
    spine.set_edgecolor('#2e7d32'); spine.set_linewidth(2)
axes[1, 3].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

# Hide any unused axes
n_bottom = len(avail_markers[3:])
for j in range(n_bottom, 3):
    axes[1, j].axis('off')

fig.suptitle(f'Region: {MOSAIC_REGION}  |  {nr}x{nc} patches  |  {len(patch_data[0]["markers"])} biomarkers',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
